# PS S6E4 - exp_20260410_033_cat_formula6_orig_pair

目的:
 - 030 の formula6 系を強化する
 - formula 系を主役に残したまま、
   original priors と pairwise TE を最小限だけ追加する

今回残すもの:
 - formula_score 系
 - threshold / score 系特徴

今回追加するもの:
 - original target prior features
 - limited pairwise columns + multiclass TE
 - original weighted merge

今回やらないもの:
 - 018 の全部盛り再現
 - heavy Optuna
 - 過剰な pairwise 拡張

## Imports

In [1]:
import os
import gc
import json
import random
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import TargetEncoder

try:
    import torch
except Exception:
    torch = None

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

## CFG

In [2]:
class CFG:
    EXP_ID = "exp_20260410_033_cat_formula6_orig_pair"
    EXP_NAME = "cat_formula6_orig_pair"
    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    ORIG_PATH = "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"

    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    SEED = 42
    N_FOLDS = 5
    TE_CV = 3
    ORIG_ROW_WEIGHT = 0.35
    N_BINS_ORIG_PRIOR = 10

    PAIR_SOURCE_COLS = [
        "Soil_Moisture",
        "Temperature_C",
        "Rainfall_mm",
        "Wind_Speed_kmh",
        "Mulching_Used",
        "Crop_Growth_Stage",
        "Season",
        "Region",
    ]
    PAIR_MIN_UNIQUE = 2
    PAIR_MAX_UNIQUE = 200
    TOPK_PAIRWISE = 24

    CATBOOST_PARAMS = {
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1:average=Macro",
        "iterations": 3000,
        "learning_rate": 0.03,
        "depth": 8,
        "grow_policy": "SymmetricTree",
        "random_seed": SEED,
        "early_stopping_rounds": 150,
        "verbose": 200,
    }

    BIAS_GRID = [
        [1.0, 1.0, 1.0],
        [1.2, 1.2, 1.2],
        [1.5, 1.5, 1.5],
        [1.5, 1.3, 1.8],
        [1.7, 1.5, 2.3],
        [1.8, 1.5, 2.4],
    ]

## base columns

In [3]:
BASE_NUM_COLS = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

BASE_CAT_COLS = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Mulching_Used",
    "Region",
]

ALL_BASE = BASE_NUM_COLS + BASE_CAT_COLS

## utility

In [4]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch is not None:
        try:
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)
        except Exception:
            pass

def maybe_set_catboost_device(params):
    params = params.copy()
    use_gpu = False
    if torch is not None:
        try:
            use_gpu = torch.cuda.is_available()
        except Exception:
            use_gpu = False

    if use_gpu:
        params["task_type"] = "GPU"
        params["devices"] = "0"
    else:
        params["task_type"] = "CPU"
        params.pop("devices", None)
    return params

def normalize_proba(p):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 1e-15, None)
    row_sum = p.sum(axis=1, keepdims=True)
    row_sum = np.where(row_sum == 0, 1.0, row_sum)
    return p / row_sum

def balanced_acc_score_mc(y_true, proba_or_pred):
    arr = np.asarray(proba_or_pred)
    if arr.ndim == 2:
        pred = np.argmax(arr, axis=1)
    else:
        pred = arr
    return float(balanced_accuracy_score(y_true, pred))

def apply_class_weights_to_proba(proba, class_weights):
    class_weights = np.asarray(class_weights, dtype=float)
    adjusted = proba * class_weights[None, :]
    return normalize_proba(adjusted)

seed_everything(CFG.SEED)
catboost_params = maybe_set_catboost_device(CFG.CATBOOST_PARAMS)

## load data

In [5]:
train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
orig = pd.read_csv(CFG.ORIG_PATH)

print("train:", train.shape)
print("test :", test.shape)
print("orig :", orig.shape)

for col in BASE_CAT_COLS + [CFG.TARGET_COL]:
    if col in train.columns:
        train[col] = train[col].astype(str)
    if col in test.columns:
        test[col] = test[col].astype(str)
    if col in orig.columns:
        orig[col] = orig[col].astype(str)

target2idx = {v: i for i, v in enumerate(sorted(train[CFG.TARGET_COL].unique()))}
idx2target = {v: k for k, v in target2idx.items()}

y = train[CFG.TARGET_COL].map(target2idx).values.astype(int)
y_orig = orig[CFG.TARGET_COL].map(target2idx).values.astype(int)

display(train.head())
print(target2idx)

train: (630000, 21)
test : (270000, 20)
orig : (10000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


{'High': 0, 'Low': 1, 'Medium': 2}


## formula6 features

In [6]:
def add_formula6_features(df):
    d = df.copy()

    d["f_low_moisture"] = (d["Soil_Moisture"] <= 25).astype(int)
    d["f_high_temp"] = (d["Temperature_C"] >= 28).astype(int)
    d["f_low_rain"] = (d["Rainfall_mm"] <= 900).astype(int)
    d["f_high_wind"] = (d["Wind_Speed_kmh"] >= 10).astype(int)
    d["f_no_mulch"] = (d["Mulching_Used"].astype(str) == "No").astype(int)
    d["f_sensitive_stage"] = d["Crop_Growth_Stage"].astype(str).isin(
        ["Sowing", "Flowering"]
    ).astype(int)

    d["formula_score"] = (
        2.0 * d["f_low_moisture"]
        + 1.5 * d["f_high_temp"]
        + 1.5 * d["f_low_rain"]
        + 1.0 * d["f_high_wind"]
        + 1.0 * d["f_no_mulch"]
        + 1.0 * d["f_sensitive_stage"]
    )

    d["score_abs"] = np.abs(d["formula_score"])
    d["low_score"] = (d["formula_score"] <= 2.0).astype(int)
    d["mid_score"] = ((d["formula_score"] > 2.0) & (d["formula_score"] < 5.0)).astype(int)
    d["high_score"] = (d["formula_score"] >= 5.0).astype(int)

    d["formula_score_x_moisture"] = d["formula_score"] * d["Soil_Moisture"]
    d["formula_score_x_rain"] = d["formula_score"] * d["Rainfall_mm"]
    d["formula_score_x_temp"] = d["formula_score"] * d["Temperature_C"]

    return d

train = add_formula6_features(train)
test = add_formula6_features(test)
orig = add_formula6_features(orig)

FORMULA_COLS = [
    "f_low_moisture",
    "f_high_temp",
    "f_low_rain",
    "f_high_wind",
    "f_no_mulch",
    "f_sensitive_stage",
    "formula_score",
    "score_abs",
    "low_score",
    "mid_score",
    "high_score",
    "formula_score_x_moisture",
    "formula_score_x_rain",
    "formula_score_x_temp",
]

print("n_formula_cols =", len(FORMULA_COLS))
display(train[FORMULA_COLS + [CFG.TARGET_COL]].head())

n_formula_cols = 14


,f_low_moisture,f_high_temp,f_low_rain,f_high_wind,f_no_mulch,f_sensitive_stage,formula_score,score_abs,low_score,mid_score,high_score,formula_score_x_moisture,formula_score_x_rain,formula_score_x_temp,Irrigation_Need
0,0,0,1,1,1,1,4.5,4.5,0,1,0,146.61,3266.955,67.545,Low
1,0,0,0,0,0,0,0.0,0.0,1,0,0,0.00,0.000,0.000,Low
2,0,0,0,0,0,0,0.0,0.0,1,0,0,0.00,0.000,0.000,Low
3,1,0,0,0,0,1,3.0,3.0,0,1,0,39.96,4071.990,39.960,Medium
4,0,0,0,1,1,1,3.0,3.0,0,1,0,177.42,4614.600,60.660,Low


## original priors

In [7]:
def _safe_multiclass_target_mean(series):
    codes, _ = pd.factorize(series.astype(str))
    return float(np.mean(codes)) if len(codes) > 0 else 0.0

def build_original_prior_mapping_for_categorical(original_df, col, target_col):
    mapping = original_df.groupby(col)[target_col].apply(_safe_multiclass_target_mean)
    global_mean = _safe_multiclass_target_mean(original_df[target_col])
    return mapping, global_mean

def build_original_prior_mapping_for_numeric(original_df, col, target_col, n_bins=10):
    x = original_df[col].astype(float)
    try:
        _, bin_edges = pd.qcut(x, q=n_bins, duplicates="drop", retbins=True)
    except ValueError:
        quantiles = np.linspace(0, 1, min(n_bins, max(2, x.nunique())) + 1)
        bin_edges = np.unique(np.quantile(x, quantiles))

    bin_edges = np.asarray(bin_edges, dtype=float)
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf

    orig_binned = pd.cut(x, bins=bin_edges, include_lowest=True)
    tmp = pd.DataFrame({col: orig_binned, target_col: original_df[target_col]})
    mapping = tmp.groupby(col, observed=False)[target_col].apply(_safe_multiclass_target_mean)
    global_mean = _safe_multiclass_target_mean(original_df[target_col])
    return bin_edges, mapping, global_mean

def add_original_target_priors(train_df, valid_df, test_df, original_df, target_col, base_cols, n_bins=10):
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    test_df = test_df.copy()

    for col in base_cols:
        feat_name = f"TE_ORIG_{col}"

        if col in BASE_CAT_COLS:
            mapping, global_mean = build_original_prior_mapping_for_categorical(
                original_df, col, target_col
            )
            train_df[feat_name] = train_df[col].map(mapping).fillna(global_mean).astype(float)
            valid_df[feat_name] = valid_df[col].map(mapping).fillna(global_mean).astype(float)
            test_df[feat_name] = test_df[col].map(mapping).fillna(global_mean).astype(float)
        else:
            bin_edges, mapping, global_mean = build_original_prior_mapping_for_numeric(
                original_df, col, target_col, n_bins=n_bins
            )

            train_bin = pd.cut(train_df[col].astype(float), bins=bin_edges, include_lowest=True)
            valid_bin = pd.cut(valid_df[col].astype(float), bins=bin_edges, include_lowest=True)
            test_bin = pd.cut(test_df[col].astype(float), bins=bin_edges, include_lowest=True)

            train_df[feat_name] = pd.Series(train_bin.astype(object), index=train_df.index).map(mapping).fillna(global_mean).astype(float)
            valid_df[feat_name] = pd.Series(valid_bin.astype(object), index=valid_df.index).map(mapping).fillna(global_mean).astype(float)
            test_df[feat_name] = pd.Series(test_bin.astype(object), index=test_df.index).map(mapping).fillna(global_mean).astype(float)

    return train_df, valid_df, test_df

## pairwise columns

In [8]:
def make_pair_name(c1, c2):
    return f"PAIR__{c1}__{c2}"

def build_pairwise_candidates(df, source_cols, min_unique=2, max_unique=200):
    pair_defs = []
    for c1, c2 in combinations(source_cols, 2):
        s = df[c1].astype(str) + "__" + df[c2].astype(str)
        nunique = s.nunique(dropna=False)
        if min_unique <= nunique <= max_unique:
            pair_defs.append((c1, c2, nunique))
    return pair_defs

def rank_pair_defs(df, pair_defs):
    rows = []
    n = len(df)
    for c1, c2, nunique in pair_defs:
        s = df[c1].astype(str) + "__" + df[c2].astype(str)
        vc = s.value_counts(dropna=False)
        rows.append({
            "col1": c1,
            "col2": c2,
            "pair_name": make_pair_name(c1, c2),
            "nunique": nunique,
            "top1_ratio": float(vc.iloc[0] / n),
            "top5_ratio": float(vc.iloc[:5].sum() / n),
            "score": float(vc.iloc[:5].sum() / n) - 0.001 * nunique,
        })
    out = pd.DataFrame(rows).sort_values(["score", "top5_ratio"], ascending=[False, False]).reset_index(drop=True)
    return out

def add_pair_columns(df, selected_pair_defs):
    d = df.copy()
    for c1, c2 in selected_pair_defs:
        name = make_pair_name(c1, c2)
        d[name] = d[c1].astype(str) + "__" + d[c2].astype(str)
    return d

pair_defs = build_pairwise_candidates(
    train,
    CFG.PAIR_SOURCE_COLS,
    min_unique=CFG.PAIR_MIN_UNIQUE,
    max_unique=CFG.PAIR_MAX_UNIQUE,
)
pair_rank_df = rank_pair_defs(train, pair_defs)
selected_pair_defs = list(zip(
    pair_rank_df.head(CFG.TOPK_PAIRWISE)["col1"],
    pair_rank_df.head(CFG.TOPK_PAIRWISE)["col2"],
))
PAIR_COLS = [make_pair_name(c1, c2) for c1, c2 in selected_pair_defs]

train = add_pair_columns(train, selected_pair_defs)
test = add_pair_columns(test, selected_pair_defs)
orig = add_pair_columns(orig, selected_pair_defs)

print("accepted_pair_count =", len(pair_defs))
print("selected_pair_count =", len(selected_pair_defs))
display(pair_rank_df.head(20))

accepted_pair_count = 6
selected_pair_count = 6


,col1,col2,pair_name,nunique,top1_ratio,top5_ratio,score
0,Mulching_Used,Season,PAIR__Mulching_Used__Season,6,0.179444,0.840100,0.834100
1,Mulching_Used,Crop_Growth_Stage,PAIR__Mulching_Used__Crop_Growth_Stage,8,0.137090,0.657800,0.649800
2,Mulching_Used,Region,PAIR__Mulching_Used__Region,10,0.109606,0.523149,0.513149
3,Crop_Growth_Stage,Season,PAIR__Crop_Growth_Stage__Season,12,0.090603,0.438851,0.426851
4,Season,Region,PAIR__Season__Region,15,0.073254,0.354475,0.339475
5,Crop_Growth_Stage,Region,PAIR__Crop_Growth_Stage__Region,20,0.056144,0.275154,0.255154


## pairwise TE helper

In [9]:
def add_pairwise_te_features(tr_df, va_df, te_df, pair_cols, target_col):
    te = TargetEncoder(target_type="multiclass", cv=CFG.TE_CV, random_state=CFG.SEED)

    tr_enc = te.fit_transform(tr_df[pair_cols], tr_df[target_col].map(target2idx))
    va_enc = te.transform(va_df[pair_cols])
    te_enc = te.transform(te_df[pair_cols])

    tr_enc = pd.DataFrame(tr_enc, index=tr_df.index)
    va_enc = pd.DataFrame(va_enc, index=va_df.index)
    te_enc = pd.DataFrame(te_enc, index=te_df.index)

    te_col_names = [f"TE_pair_{i}" for i in range(tr_enc.shape[1])]
    tr_enc.columns = te_col_names
    va_enc.columns = te_col_names
    te_enc.columns = te_col_names

    tr_out = pd.concat([tr_df.reset_index(drop=True), tr_enc.reset_index(drop=True)], axis=1)
    va_out = pd.concat([va_df.reset_index(drop=True), va_enc.reset_index(drop=True)], axis=1)
    te_out = pd.concat([te_df.reset_index(drop=True), te_enc.reset_index(drop=True)], axis=1)

    return tr_out, va_out, te_out, te_col_names

## feature groups

In [10]:
ORIG_PRIOR_SOURCE_COLS = ALL_BASE + ["formula_score", "score_abs"]

FORMULA_CAT_COLS = [
    "f_low_moisture",
    "f_high_temp",
    "f_low_rain",
    "f_high_wind",
    "f_no_mulch",
    "f_sensitive_stage",
    "low_score",
    "mid_score",
    "high_score",
]

FORMULA_NUM_COLS = [
    "formula_score",
    "score_abs",
    "formula_score_x_moisture",
    "formula_score_x_rain",
    "formula_score_x_temp",
]

BASE_FEATURES_033 = BASE_NUM_COLS + BASE_CAT_COLS + FORMULA_COLS
print("n_base_features_033 =", len(BASE_FEATURES_033))

n_base_features_033 = 33


## CV main

In [11]:
skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

oof_proba = np.zeros((len(train), 3), dtype=float)
test_pred_sum = np.zeros((len(test), 3), dtype=float)
fold_scores = []
feature_importances = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y), start=1):
    print("=" * 100)
    print(f"Fold {fold}/{CFG.N_FOLDS}")
    print("=" * 100)

    tr = train.iloc[tr_idx].copy().reset_index(drop=True)
    va = train.iloc[va_idx].copy().reset_index(drop=True)
    te = test.copy().reset_index(drop=True)

    tr, va, te = add_original_target_priors(
        tr, va, te,
        original_df=orig,
        target_col=CFG.TARGET_COL,
        base_cols=ORIG_PRIOR_SOURCE_COLS,
        n_bins=CFG.N_BINS_ORIG_PRIOR,
    )

    tr, va, te, te_col_names = add_pairwise_te_features(
        tr_df=tr,
        va_df=va,
        te_df=te,
        pair_cols=PAIR_COLS,
        target_col=CFG.TARGET_COL,
    )

    # original weighted merge
    orig_aug = orig.copy().reset_index(drop=True)
    orig_aug, _, _ = add_original_target_priors(
        orig_aug.copy(),
        orig_aug.copy(),
        orig_aug.drop(columns=[CFG.TARGET_COL]).copy(),
        original_df=orig,
        target_col=CFG.TARGET_COL,
        base_cols=ORIG_PRIOR_SOURCE_COLS,
        n_bins=CFG.N_BINS_ORIG_PRIOR,
    )
    orig_aug = add_pair_columns(orig_aug, selected_pair_defs)
    orig_aug_enc = TargetEncoder(target_type="multiclass", cv=CFG.TE_CV, random_state=CFG.SEED)
    orig_pair_enc = orig_aug_enc.fit_transform(orig_aug[PAIR_COLS], orig_aug[CFG.TARGET_COL].map(target2idx))
    orig_pair_enc = pd.DataFrame(orig_pair_enc, index=orig_aug.index)
    orig_pair_enc.columns = te_col_names
    orig_aug = pd.concat([orig_aug.reset_index(drop=True), orig_pair_enc.reset_index(drop=True)], axis=1)

    feature_cols = BASE_FEATURES_033 + [f"TE_ORIG_{c}" for c in ORIG_PRIOR_SOURCE_COLS] + te_col_names
    feature_cols = list(dict.fromkeys(feature_cols))

    cat_feature_cols = BASE_CAT_COLS + FORMULA_CAT_COLS
    cat_feature_cols = [c for c in cat_feature_cols if c in feature_cols]

    X_tr = pd.concat([
        tr[feature_cols].copy(),
        orig_aug[feature_cols].copy(),
    ], axis=0, ignore_index=True)

    y_tr = np.concatenate([
        tr[CFG.TARGET_COL].map(target2idx).values.astype(int),
        orig_aug[CFG.TARGET_COL].map(target2idx).values.astype(int),
    ])

    sample_weight = np.concatenate([
        np.ones(len(tr), dtype=float),
        np.full(len(orig_aug), CFG.ORIG_ROW_WEIGHT, dtype=float),
    ])

    X_va = va[feature_cols].copy()
    y_va = va[CFG.TARGET_COL].map(target2idx).values.astype(int)
    X_te = te[feature_cols].copy()

    cat_feature_indices = [X_tr.columns.get_loc(c) for c in cat_feature_cols]

    model = CatBoostClassifier(**catboost_params)
    model.fit(
        Pool(X_tr, y_tr, cat_features=cat_feature_indices, weight=sample_weight),
        eval_set=Pool(X_va, y_va, cat_features=cat_feature_indices),
        use_best_model=True,
    )

    va_pred = model.predict_proba(X_va)
    te_pred = model.predict_proba(X_te)

    oof_proba[va_idx] = va_pred
    test_pred_sum += te_pred
    fold_score = balanced_acc_score_mc(y_va, va_pred)
    fold_scores.append(float(fold_score))
    print(f"fold_ba = {fold_score:.6f}")

    fi = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.get_feature_importance(),
        "fold": fold,
    })
    feature_importances.append(fi)

    del tr, va, te, X_tr, X_va, X_te, model
    gc.collect()

test_proba = test_pred_sum / CFG.N_FOLDS
raw_cv_ba = balanced_acc_score_mc(y, oof_proba)
print("raw_cv_ba =", raw_cv_ba)

Fold 1/5
0:	learn: 0.9675607	test: 0.9675467	best: 0.9675467 (0)	total: 268ms	remaining: 13m 25s
200:	learn: 0.9696716	test: 0.9697948	best: 0.9698954 (186)	total: 5.66s	remaining: 1m 18s
400:	learn: 0.9708174	test: 0.9700500	best: 0.9701839 (307)	total: 10.7s	remaining: 1m 9s
bestTest = 0.9701838638
bestIteration = 307
Shrink model to first 308 iterations.
fold_ba = 0.960207
Fold 2/5
0:	learn: 0.9677069	test: 0.9679547	best: 0.9679547 (0)	total: 52.4ms	remaining: 2m 37s
200:	learn: 0.9699017	test: 0.9689184	best: 0.9692506 (171)	total: 5.55s	remaining: 1m 17s
400:	learn: 0.9708642	test: 0.9695202	best: 0.9695202 (382)	total: 10.4s	remaining: 1m 7s
600:	learn: 0.9715386	test: 0.9697119	best: 0.9697526 (581)	total: 15.1s	remaining: 1m
800:	learn: 0.9719765	test: 0.9698287	best: 0.9699160 (772)	total: 19.7s	remaining: 54s
1000:	learn: 0.9725125	test: 0.9699651	best: 0.9700116 (999)	total: 24.3s	remaining: 48.6s
bestTest = 0.9700116244
bestIteration = 999
Shrink model to first 1000 iterat

## bias tuning

In [12]:
best_score = raw_cv_ba
best_class_weights = np.ones(3, dtype=float)

for cw in CFG.BIAS_GRID:
    cw = np.asarray(cw, dtype=float)
    adjusted = apply_class_weights_to_proba(oof_proba, cw)
    score = balanced_acc_score_mc(y, adjusted)
    if score > best_score:
        best_score = score
        best_class_weights = cw.copy()

oof_proba_biased = apply_class_weights_to_proba(oof_proba, best_class_weights)
test_proba_biased = apply_class_weights_to_proba(test_proba, best_class_weights)

print("best_class_weights =", best_class_weights.tolist())
print("biased_cv_ba =", best_score)
print("improvement =", best_score - raw_cv_ba)

best_class_weights = [1.0, 1.0, 1.0]
biased_cv_ba = 0.9613632898395311
improvement = 0.0


## submission

In [13]:
submission = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv")
submission[CFG.TARGET_COL] = np.argmax(test_proba_biased, axis=1)
submission[CFG.TARGET_COL] = submission[CFG.TARGET_COL].map(idx2target)
submission.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

display(submission.head())

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


## save artifacts

In [14]:
feature_importance_df = pd.concat(feature_importances, axis=0, ignore_index=True)
feature_importance_mean = (
    feature_importance_df.groupby("feature", as_index=False)["importance"]
    .mean()
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba.npy", oof_proba)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba.npy", test_proba)
np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba_biased.npy", oof_proba_biased)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba_biased.npy", test_proba_biased)

feature_importance_df.to_csv(CFG.SAVE_DIR / "feature_importance_by_fold.csv", index=False)
feature_importance_mean.to_csv(CFG.SAVE_DIR / "feature_importance.csv", index=False)
pair_rank_df.to_csv(CFG.SAVE_DIR / "pair_rank_df.csv", index=False)

## save cv_result.json

In [15]:
cv_result = {
    "exp_id": CFG.EXP_ID,
    "raw_cv_ba": float(raw_cv_ba),
    "biased_cv_ba": float(best_score),
    "fold_scores": fold_scores,
    "best_class_weights": best_class_weights.tolist(),
    "n_formula_features": len(FORMULA_COLS),
    "n_pair_columns": len(PAIR_COLS),
    "pair_columns": PAIR_COLS,
    "n_features_total": int(len(feature_importance_mean)),
}

with open(CFG.SAVE_DIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(cv_result, f, ensure_ascii=False, indent=2)

print("saved to:", CFG.SAVE_DIR)
display(feature_importance_mean.head(30))

saved to: /kaggle/working/exp_20260410_033_cat_formula6_orig_pair


,feature,importance
0,TE_pair_3,17.152823
1,f_low_moisture,11.416246
2,Temperature_C,9.123968
3,f_high_wind,8.848882
4,TE_pair_4,7.805527
5,Mulching_Used,7.749914
6,Soil_Moisture,6.646714
7,TE_pair_5,2.806294
8,TE_pair_15,2.632746
9,Rainfall_mm,2.570458


## quick summary

In [16]:
summary_df = pd.DataFrame({
    "item": [
        "raw_cv_ba",
        "biased_cv_ba",
        "improvement",
        "n_formula_features",
        "n_pair_columns",
        "n_features_total",
    ],
    "value": [
        raw_cv_ba,
        best_score,
        best_score - raw_cv_ba,
        len(FORMULA_COLS),
        len(PAIR_COLS),
        len(feature_importance_mean),
    ],
})
display(summary_df)

,item,value
0,raw_cv_ba,0.961363
1,biased_cv_ba,0.961363
2,improvement,0.000000
3,n_formula_features,14.000000
4,n_pair_columns,6.000000
5,n_features_total,72.000000
